# Implementing an LR-Table-Generator

## A Grammar for Grammars

We use the following grammar to describe the syntax of a context free grammar.
Since <span style="font-variant:small-caps;">Larkita</span> supports *EBNF*, we can use the postfix operators
`*` (zero or more repetitions) and `+` (one or more repetitions) and we can group symbols with parentheses.
This way, the grammar is considerably shorter than it would be in plain *BNF*:
```
grammar : rule+
rule    : RULE_NAME ":" body ("|" body)*
body    : item*
item    : VARIABLE | TOKEN | LITERAL
```
Here, `RULE_NAME`, `VARIABLE`, `TOKEN`, and `LITERAL` are *terminals*:
- a `VARIABLE` is a *syntactical variable* and is written in lower case letters,
- a `RULE_NAME` is a variable that is followed by a colon, i.e. the variable that is defined by a rule,
- a `TOKEN` is the name of a token and is written in upper case letters, while
- a `LITERAL` is a string enclosed in double quotes.

This is also the notation of the grammars in the directory `Examples`.  A rule is not terminated by a semicolon:
It ends where the next rule begins, i.e. at the next variable that is followed by a colon.

As the goal is to generate an *LR-table-generator* we first need to implement a parser for context free grammars.
The file `cc.g` in the directory `Examples` contains a small example grammar.

In [ ]:
from IPython.display import Code, display

Code(filename='Examples/cc.g', language='larkita')

We use <span style="font-variant:small-caps;">Larkita</span> to develop a parser for context free grammars.

In [ ]:
from pprint import pprint

from larkita import Larkita, UnexpectedInput

The string `grammar_of_grammars` below is the *EBNF grammar* that describes context free grammars.
The following conventions of <span style="font-variant:small-caps;">Larkita</span> are important:
- Names written in lower case letters are *grammar rules*, while names written in upper case letters are
  *terminals*, i.e. they are the rules that are used by the scanner.
- Terminals are defined either as strings or as *regular expressions* that are enclosed in forward slashes.
- The parser returns the *abstract syntax tree* of a grammar as a nested tuple.  The string literals of an
  alternative form the *head* of the tuple that is returned for this alternative, while all other symbols become
  its children.  The strings `":"` and `"|"` occurring in the rule `rule` carry no information, so the
  annotation `%ignore ":", "|"` removes them from the head.  As no literal is left, the name of the rule is
  used as head:  A rule yields the tuple `('rule', variable, body_1, ..., body_n)` and a body yields the tuple
  `('body', item_1, ..., item_n)`.
- A token is represented as a pair consisting of the name of the token and its text, for example
  `('VARIABLE', 'expr')`.  As every alternative of the rule `item` consists of a single symbol, `item` creates no
  tuple of its own.  Instead, the tokens show up as immediate children of the tuple `('body', ...)`.
- The rules of a grammar are not terminated by a semicolon.  Therefore, after the body of a rule, a variable can
  either be the next item of this body or the beginning of the next rule.  Only the colon that follows the variable
  tells these cases apart, but an LR(1) parser sees only one token ahead.  Hence, the scanner recognizes a variable
  that is followed by a colon as a token of its own, `RULE_NAME`.  Its regular expression ends with the *lookahead*
  `(?=[ \t]*:)`, which checks that a colon follows without making the colon part of the token.  As both
  `RULE_NAME` and `VARIABLE` match the name of the variable, the priority `2` in `RULE_NAME.2` tells the scanner to
  prefer `RULE_NAME`.
- The directive `%ignore` lists those terminals that the scanner discards.  We ignore both white space and
  comments.  Note that the two slashes that start a comment have to be escaped in the regular expression
  `/\/\/[^\n]*/` because in a grammar the string `//` starts a comment.

In [ ]:
grammar_of_grammars = r"""
    grammar : rule+                                 // a grammar is a non-empty list of rules
    rule    : RULE_NAME ":" body ("|" body)*        %ignore ":", "|"
    body    : item*                                 // a body can be empty
    item    : VARIABLE | TOKEN | LITERAL

    RULE_NAME.2 : /[a-z][a-z0-9_]*(?=[ \t]*:)/      // a variable that is followed by a colon
    VARIABLE    : /[a-z][a-z0-9_]*/
    TOKEN       : /[A-Z][A-Z0-9_]*/
    LITERAL     : /".*?"/
    COMMENT     : /\/\/[^\n]*/

    %import common.WS
    %ignore WS, COMMENT
"""

The class `Larkita` creates both the scanner and the parser.  The keyword argument `start` specifies the start
symbol.  It could be omitted here, since the start symbol defaults to the first rule of the grammar.  By default,
`Larkita` generates a *canonical LR(1) parser* and writes a description of its states into the file given by
`out_file`.  The keyword argument `lexer='basic'` selects a scanner that works independently of the parser.  This
is not necessary for parsing, but it enables us to inspect the token stream via the method `lex` that is used in the
function `test_scanner` below.

In [ ]:
parser = Larkita(grammar_of_grammars, start='grammar', lexer='basic', out_file='grammar-of-grammars.out')

Given the name of a file containing a grammar, the function `test_scanner` prints the tokens that
the scanner produces.  Every token `t` provides the attributes `t.type`, `t.value`, `t.line`, and `t.column`.

In [ ]:
def test_scanner(file_name: str) -> None:
    with open(file_name, 'r') as handle:
        program = handle.read() 
    display(Code(program, language='larkita'))
    for t in parser.lex(program):           # start scanning and collect all tokens
        print(f'line {t.line:>2}, column {t.column:>2}: {t.type:<10}{t.value}')

In [ ]:
test_scanner('Examples/cc.g')

The method `parse` of the object `parser` returns the *abstract syntax tree* as a nested tuple, which we print with
the function `pprint`.  Note that the literals `":"` and `"|"` do not occur in this tuple and that there are
no tuples labelled `item`.

In [ ]:
with open('Examples/cc.g', 'r') as handle:
    pprint(parser.parse(handle.read()))

The nested tuple is not a convenient representation for the generation of an *LR-table*.  Therefore,
we transform it into a list of tuples.  Below, we define a few *type aliases* in order to make the types of
this transformation more readable.

In [ ]:
Variable = str
Token    = str
Symbol   = Variable | Token
Symbols  = tuple[Symbol, ...]

The function `grammar_to_rules` converts the nested tuple into this representation.  A rule of the form
`v : b1 | b2 | b3` is represented by the tuple `('rule', ('RULE_NAME', 'v'), body1, body2, body3)` and defines three
grammar rules.  The function creates the tuple `(v,) + β` for every body `('body', item_1, ..., item_n)` of every rule,
where `β` is the tuple of the texts of the items.  The result has the type `list[Symbols]`.

In [ ]:
def grammar_to_rules(ast: tuple) -> list[Symbols]:
    _, *rules = ast
    return [ (variable,) + tuple(text for _, text in items)
             for _, (_, variable), *bodies in rules
             for _, *items in bodies ]

The function `parse` takes the name of a file containing a grammar and returns the list of the
grammar rules that are defined in this file.  If the file contains a syntax error, the parser raises an
exception of the class `UnexpectedInput`.  This class is the common superclass of the exception
`UnexpectedCharacters`, which is raised by the scanner, and the exception `UnexpectedToken`, which is raised
by the parser.  Both of these exceptions know the line and the column of the error, while the method
`get_context` returns the offending line together with a marker pointing at the error.

In [ ]:
def parse(file: str) -> list[Symbols]:
    with open(file, 'r') as handle:
        grammar = handle.read() 
    display(Code(grammar, language='larkita'))
    try:
        ast = parser.parse(grammar)
    except UnexpectedInput as error:
        print(f'Syntax error in line {error.line}, column {error.column}:')
        print(error.get_context(grammar))
        raise
    return grammar_to_rules(ast)

In [ ]:
ruleList = parse('Examples/cc.g')
ruleList

The parser will return a list of grammar rules, where each rule of the form
$$ a \rightarrow \beta $$
is stored as the tuple `(a,) + 𝛽`.

### The Class `GrammarRule`

In [ ]:
from dataclasses import dataclass

The class `GrammarRule` is used to store a single grammar rule.  As we have to use objects of type `GrammarRule` as *keys* in a dictionary later, they have to be hashable.  Since `GrammarRule` is a *frozen* dataclass, Python generates the methods `__eq__` and `__hash__` automatically.  Therefore, we can store objects of the class in a `set`.

In [ ]:
@dataclass(frozen=True, repr=False)
class GrammarRule:
    mVariable: Variable
    mBody:     Symbols

    def __repr__(self) -> str:
        return f'{self.mVariable} → {" ".join(self.mBody)}'

The function `transform(rules)` takes a list of tuples representing grammar rules and converts these tuples into objects of class `GrammarRule`.

In [ ]:
def transform(rules: list[tuple[Symbol, ...]]) -> list[GrammarRule]:
    return [ GrammarRule(var, tuple(body)) for (var, *body) in rules]

In [ ]:
grammar = transform(ruleList)
grammar

Given a string `name`, which is either a *variable*, a *token*, or a *literal*, the function `is_var` checks whether `name` is a variable.  The function can distinguish variable names from tokens and literals because variable names consist only of lower case letters, while tokens are all uppercase and literals start with a double quote.

In [ ]:
def is_var(name: Symbol) -> bool:
    return name[0].islower()

Given a list `Rules` of `GrammarRules`, the function `collect_variables(Rules)` returns the set of all *variables* that occur on the left hand sides of the rules in `Rules`.

In [ ]:
def collect_variables(Rules: list[GrammarRule]) -> set[Variable]:
    return { rule.mVariable for rule in Rules }

Given a list `Rules` of `GrammarRules`, the function `collect_tokens(Rules)` returns the set of all *tokens* and *literals* occurring in `Rules`.

In [ ]:
def collect_tokens(Rules: list[GrammarRule]) -> set[Token]:
    return  { item for rule in Rules
                   for item in rule.mBody
                   if not is_var(item)
            }

## Marked Rules and Extended Marked Rules

The class `MarkedRule` stores a single *marked rule* of the form
$$ v \rightarrow \alpha \bullet \beta $$
where the *variable* $v$ is stored in the member variable `mVariable`, while $\alpha$ and $\beta$ are stored in the variables `mAlpha`and `mBeta` respectively.  These variables are assumed to contain tuples of *grammar symbols*.  A *grammar symbol* is either
- a *variable*,
- a *token*, or
- a *literal*, i.e. a string enclosed in double quotes.


The states of an LR parser do not consist of marked rules but of *extended marked rules*, which are defined below.
We need marked rules only as the *cores* of extended marked rules:  The function `combine_rules`, which is defined
below, uses the cores as keys of a dictionary.  Therefore, marked rules have to be hashable.  As `MarkedRule` is a
*frozen* dataclass, Python generates the methods `__eq__` and `__hash__` automatically.

In [ ]:
@dataclass(frozen=True, repr=False)
class MarkedRule():
    mVariable: Variable
    mAlpha:    Symbols
    mBeta:     Symbols

    def __repr__(self) -> str:
        return ' '.join([self.mVariable, '→', *self.mAlpha, '•', *self.mBeta])

The class `ExtendedMarkedRule` stores a single *extended marked rule* of the form
$$ v \rightarrow \alpha \bullet \beta : L $$
where $v$, $\alpha$, and $\beta$ are stored in the member variables `mVariable`, `mAlpha`, and `mBeta` as in the
class `MarkedRule`.  The set of follow tokens $L$ is stored in the member variable `mFollow` as a `frozenset` of tokens.

Later, we need to maintain sets of *extended marked rules* to represent *states*.  Therefore, extended marked rules
have to be hashable, too.  As `ExtendedMarkedRule` is a *frozen* dataclass, Python generates the methods `__eq__` and
`__hash__` automatically.  This is the reason why `mFollow` is a `frozenset`:  A `set` is not hashable.

The methods that are only declared in the class body, i.e. with the body `...`, are implemented in the
cells below and attached to the class afterwards.  These declarations tell the type checker which
methods and member variables the class provides.

In [ ]:
@dataclass(frozen=True, repr=False)
class ExtendedMarkedRule():
    mVariable: Variable
    mAlpha:    Symbols
    mBeta:     Symbols
    mFollow:   frozenset[Token]

    def __repr__(self) -> str:
        if len(self.mFollow) > 1:
            followStr = '{' + ','.join(sorted(self.mFollow)) + '}'
        else:
            followStr = ','.join(self.mFollow)
        return ' '.join([self.mVariable, '→', *self.mAlpha, '•', *self.mBeta]) + f': {followStr}'

    def is_complete(self) -> bool: ...
    def symbol_after_dot(self) -> Symbol | None: ...
    def next_var(self) -> Variable | None: ...
    def move_dot(self) -> ExtendedMarkedRule: ...
    def to_rule(self) -> GrammarRule: ...
    def to_marked_rule(self) -> MarkedRule: ...

Given an *extended marked rule* `self`, the function `is_complete` checks, whether the *extended marked rule* `self` has the form
$$ c \rightarrow \alpha\; \bullet: L,$$
i.e. it checks, whether the $\bullet$ is at the end of the grammar rule.

In [ ]:
def is_complete(self: ExtendedMarkedRule) -> bool:
    return len(self.mBeta) == 0

ExtendedMarkedRule.is_complete = is_complete
del is_complete

Given an *extended marked rule* `self` of the form
$$ c \rightarrow \alpha \bullet X\, \delta: L, $$
the function `symbol_after_dot` returns the *symbol* $X$. If there is no symbol after the $\bullet$, the method returns `None`.

In [ ]:
def symbol_after_dot(self: ExtendedMarkedRule) -> Symbol | None:
    if len(self.mBeta) > 0:
        return self.mBeta[0]
    return None

ExtendedMarkedRule.symbol_after_dot = symbol_after_dot
del symbol_after_dot

Given an extended marked rule of the form
$$ c \rightarrow \alpha \bullet b \delta: L, $$
this function returns the variable $b$ following the dot.  If there is no variable following the dot, the function returns `None`.

In [ ]:
def next_var(self: ExtendedMarkedRule) -> Variable | None:
    if len(self.mBeta) > 0:
        var = self.mBeta[0]
        if is_var(var):
            return var
    return None

ExtendedMarkedRule.next_var = next_var
del next_var

The function `move_dot(self)` transforms an *extended marked rule*  of the form 
$$ c \rightarrow \alpha \bullet X\, \beta: L $$
into an *extended marked rule* of the form
$$ c \rightarrow \alpha\, X \bullet \beta: L, $$
i.e. the $\bullet$ is moved over the next symbol.  Invocation of this method assumes that there is a symbol
following the $\bullet$.

In [ ]:
def move_dot(self: ExtendedMarkedRule) -> ExtendedMarkedRule:
    return ExtendedMarkedRule(self.mVariable, 
                              self.mAlpha + (self.mBeta[0],), 
                              self.mBeta[1:],
                              self.mFollow)

ExtendedMarkedRule.move_dot = move_dot
del move_dot

The function `to_rule(self)` turns the *extended marked rule* `self` into  a `GrammarRule`, i.e. the *extended marked rule*
$$ c \rightarrow \alpha \bullet \beta: L $$
is turned into the grammar rule
$$ c \rightarrow \alpha\, \beta. $$

In [ ]:
def to_rule(self: ExtendedMarkedRule) -> GrammarRule:
    return GrammarRule(self.mVariable, self.mAlpha + self.mBeta)

ExtendedMarkedRule.to_rule = to_rule
del to_rule

The function `to_marked_rule(self)` turns the *extended marked rule* `self` into a `MarkedRule`, i.e. the *extended marked rule*
$$ c \rightarrow \alpha \bullet \beta: L $$
is turned into the marked rule
$$ c \rightarrow \alpha\bullet \beta. $$

In [ ]:
def to_marked_rule(self: ExtendedMarkedRule) -> MarkedRule:
    return MarkedRule(self.mVariable, self.mAlpha, self.mBeta)

ExtendedMarkedRule.to_marked_rule = to_marked_rule
del to_marked_rule

Given a set of *extended marked rules* `M`, the function `combine_rules` *combines* those extended marked rules that have the same *core*:  If 
$$ a \rightarrow \beta \bullet \gamma : L $$
is an extended marked rule, then its *core* is defined as the marked rule
$$ a \rightarrow \beta \bullet \gamma. $$
If $a \rightarrow \beta \bullet \gamma : L_1$ and $a \rightarrow \beta \bullet \gamma : L_2$ are two extended marked rules, then they can be *combined* into the rule
$$ a \rightarrow \beta \bullet \gamma : L_1\cup L_2. $$
The function returns the combined rules as a list.  The dictionary `Follow` maps every core to the union of the
follow tokens of the extended marked rules with this core.  As a dictionary remembers the order in which its keys
have been inserted, the combined rule takes the place of the first extended marked rule with this core.

In [ ]:
def combine_rules(M: list[ExtendedMarkedRule]) -> list[ExtendedMarkedRule]:
    Follow: dict[MarkedRule, frozenset[Token]] = {}   # maps a core to the union of its follow tokens
    for emr in M:
        core = emr.to_marked_rule()
        Follow[core] = Follow.get(core, frozenset()) | emr.mFollow
    return [ ExtendedMarkedRule(core.mVariable, core.mAlpha, core.mBeta, L) for core, L in Follow.items() ]

## States

A *state* of the parser is a set of *extended marked rules*.  If we represented a state as a `frozenset`, the extended marked rules of a
state would be printed in an arbitrary order, since sets do not remember the order in which their elements have been
inserted.  We would rather see them in the order in which they have been added to the state: first the extended marked rules
that result from moving the dot, then the extended marked rules that have been added by the closure.  Therefore, the class
`State` stores the extended marked rules of a state twice:
- `mItems` is a tuple that contains the extended marked rules in the order of their insertion.  It is used whenever we iterate
  over the extended marked rules of a state or print them.
- `mKey` is the `frozenset` of these extended marked rules.  Only this set is used when two states are compared and when the hash
  value of a state is computed:  Two states that contain the same extended marked rules in a different order are the same state.

The constructor takes the extended marked rules in the order of their insertion.  The expression `dict.fromkeys(items)` creates
a dictionary whose keys are these extended marked rules.  As a dictionary remembers the order in which its keys have been
inserted, this removes duplicates while keeping the order.

In [ ]:
from collections.abc import Iterable, Iterator

class State:
    def __init__(self, items: Iterable[ExtendedMarkedRule]) -> None:
        self.mItems: tuple[ExtendedMarkedRule, ...] = tuple(dict.fromkeys(items))
        self.mKey  : frozenset[ExtendedMarkedRule]  = frozenset(self.mItems)

    def __eq__(self, other: object) -> bool:
        return isinstance(other, State) and self.mKey == other.mKey

    def __hash__(self) -> int:
        return hash(self.mKey)

    def __iter__(self) -> Iterator[ExtendedMarkedRule]:
        return iter(self.mItems)

    def __len__(self) -> int:
        return len(self.mItems)

    def __repr__(self) -> str:
        return '{' + ', '.join(map(str, self.mItems)) + '}'

## LR-Table-Generation

The class `Grammar` represents a context free grammar.  It stores a list of the `GrammarRules` of the given grammar.
Each grammar rule is of the form
$$ a \rightarrow \beta $$
where $\beta$ is a tuple of variables, tokens, and literals.
The start symbol is assumed to be the variable on the left hand side of the first rule. The grammar is *augmented* with the rule
$$ \widehat{s} \rightarrow s. $$
Here $s$ is the start variable of the given grammar and $\widehat{s}$ is a new variable that is the start variable of the *augmented grammar*. The symbol `$` denotes the end of input.  The non-obvious member variables of the class `Grammar` have the following interpretation
- `mStates` is the list of all states of the *LR-parser*, ordered by their names.  These states are objects
  of the class `State`.
- `mStateNames` is a dictionary assigning names of the form `s0`, `s1`, $\cdots$, `sn` to the states stored in 
  `mStates`.  The functions `action` and `goto` will be defined for *state names*, not for *states*, because 
  otherwise the table representing these functions would become both huge and unreadable.
  Therefore, the dictionary `mStateNames` is needed to associate the states with their names.
- `mConflicts` is a Boolean variable that will be set to `True` if the table generation discovers 
  a *shift/reduce conflict*, a *reduce/reduce conflict*, or a conflict between *accept* and *reduce*.

The methods that are only declared in the class body, i.e. with the body `...`, are implemented in the
cells below and attached to the class afterwards.  These declarations tell the type checker which
methods and member variables the class provides.

In [ ]:
class Grammar():
    def __init__(self, Rules: list[GrammarRule]):
        self.mRules     : list[GrammarRule] = list(Rules)  # a copy, since the grammar is augmented below
        self.mStart     : Variable          = Rules[0].mVariable
        self.mVariables : set[Variable]     = collect_variables(Rules)
        self.mTokens    : set[Token]        = collect_tokens(Rules)
        self.mStates    : list[State]       = []
        self.mStateNames: dict[State, str]  = {}
        self.mConflicts : bool              = False
        self.mVariables.add('ŝ')
        self.mTokens.add('$') # short for EOF
        self.mRules.append(GrammarRule('ŝ', (self.mStart, ))) # augment the grammar
        self.compute_tables()

    mFirst      : dict[Variable, set[Token]]
    mRuleNames  : dict[GrammarRule, str]
    mActionTable: dict[tuple[str, Token], str | tuple[str, str]]
    mGotoTable  : dict[tuple[str, Variable], str]

    def compute_tables(self) -> None: ...
    def compute_rule_names(self) -> None: ...
    def compute_first(self) -> None: ...
    def first_list(self, alpha: Symbols) -> set[Token]: ...
    def cmp_closure(self, Marked_Rules: Iterable[ExtendedMarkedRule]) -> list[ExtendedMarkedRule]: ...
    def goto(self, state: State, x: Symbol) -> State: ...
    def all_states(self) -> None: ...
    def compute_action_table(self) -> None: ...
    def compute_goto_table(self) -> None: ...
    def dump_parse_table(self, file: str) -> None: ...

Given a set of `Variables`, the function `initialize_dictionary` returns a dictionary that assigns the empty set to all variables.
This function is needed to initialize the member variable `mFirst` that is a dictionary storing the *first-sets* of
the syntactical variables.

In [ ]:
def initialize_dictionary(Variables: set[Variable]) -> dict[Variable, set[Token]]:
    return { a: set() for a in Variables }

Given a `Grammar`, the function `compute_tables` computes
- the sets `First(v)` for every variable `v`,
- the set of all *states* of the *LR-Parser*,
- the *action table*, and
- the *goto table*. 

Given a grammar `g`, the set `g.mFirst` is a dictionary such that `g.mFirst[a] = First(a)` for all variables `a`.
In contrast to an SLR parser, an LR parser does not need the sets `Follow(a)`, since the follow tokens are stored
in the extended marked rules.

In [ ]:
def compute_tables(self: Grammar) -> None:
    self.mFirst  = initialize_dictionary(self.mVariables)
    self.compute_first()
    self.compute_rule_names()
    self.all_states()
    self.compute_action_table()
    self.compute_goto_table()
    
Grammar.compute_tables = compute_tables
del compute_tables

The function `compute_rule_names` assigns a unique name to each *rule* of the grammar.  These names are used later
to represent *reduce actions* in the *action table*.

In [ ]:
def compute_rule_names(self: Grammar) -> None:
    self.mRuleNames = {}
    counter = 0
    for rule in self.mRules:
        self.mRuleNames[rule] = 'r' + str(counter)
        counter += 1
        
Grammar.compute_rule_names = compute_rule_names
del compute_rule_names

The function `compute_first(self)` computes the sets $\texttt{First}(c)$ for all variables $c$ and stores them in the dictionary `mFirst`.  Abstractly, given a variable $c$ the function $\texttt{First}(c)$ is the set of all tokens that can start a string that is derived from $c$:
$$\texttt{First}(\texttt{c}) := 
  \Bigl\{ t \in T \Bigm| \exists \gamma \in (V \cup T)^*: \texttt{c} \Rightarrow^* t\,\gamma \Bigr\}.
$$
In order to keep track of the variables that can derive the empty string $\lambda$, the implementation adds
$\lambda$, which is represented as the empty string `''`, to the set $\texttt{First}(\texttt{c})$ if
$\texttt{c} \Rightarrow^* \lambda$.  The definition of the function $\texttt{First}()$ is extended to strings
from $(V \cup T)^*$ as follows:
- $\texttt{FirstList}(\lambda) = \{\lambda\}$.
- $\texttt{FirstList}(t \beta) = \{ t \}$  if $t \in T$.
- $\texttt{FirstList}(\texttt{a} \beta) = \left\{
       \begin{array}[c]{ll}
         \bigl(\texttt{First}(\texttt{a}) - \{\lambda\}\bigr) \cup \texttt{FirstList}(\beta) & \mbox{if $\lambda \in \texttt{First}(\texttt{a})$;} \\
         \texttt{First}(\texttt{a})                                & \mbox{otherwise.}
       \end{array}
       \right.
      $ 

Hence, $\lambda \in \texttt{FirstList}(\alpha)$ holds if and only if $\alpha \Rightarrow^* \lambda$.
If $\texttt{a}$ is a variable of $G$ and the rules defining $\texttt{a}$ are given as 
$$\texttt{a} \rightarrow \alpha_1 \mid \cdots \mid \alpha_n, $$
then we have
$$\texttt{First}(\texttt{a}) = \bigcup\limits_{i=1}^n \texttt{FirstList}(\alpha_i). $$
The dictionary `mFirst` that stores this function is computed via a *fixed point iteration*.

In [ ]:
def compute_first(self: Grammar) -> None:
    change = True
    while change:
        change = False
        for rule in self.mRules:
            a, body = rule.mVariable, rule.mBody
            first_body = self.first_list(body)
            if not (first_body <= self.mFirst[a]):
                change = True
                self.mFirst[a] |= first_body
    print('First sets:')
    for v in sorted(self.mVariables):
        print(f'First({v}) = {{{", ".join(map(repr, sorted(self.mFirst[v])))}}}')
        
Grammar.compute_first = compute_first
del compute_first

Given a tuple of variables and tokens `alpha`, the function `first_list(alpha)` computes the function $\texttt{FirstList}(\alpha)$ that has been defined above.  If `alpha` is *nullable*, then the result will contain the empty string $\lambda$.

In [ ]:
def first_list(self: Grammar, alpha: Symbols) -> set[Token]:
    if len(alpha) == 0:
        return { '' }
    elif is_var(alpha[0]): 
        v, *r = alpha
        return eps_union(self.mFirst[v], self.first_list(tuple(r)))
    else:
        t = alpha[0]
        return { t }
    
Grammar.first_list = first_list
del first_list

The arguments `S` and `T` of `eps_union` are sets that contain tokens and, additionally, they might contain the empty string
$\lambda =$ `''`.  The specification of `eps_union` is:
$$ \mathtt{eps\_union}(S, T) = \left\{ \begin{array}{ll}
                                       S          & \mbox{if $\lambda \not\in S$} \\
                                       S \cup T   & \mbox{if $\lambda \in S \wedge \lambda \in T$} \\
                                       S \cup T - \{\lambda \} & \mbox{if $\lambda \in S \wedge \lambda \not\in T$}
                                      \end{array}
                              \right.
$$

In [ ]:
def eps_union(S: set[Token], T: set[Token]) -> set[Token]:
    if '' in S: 
        if '' in T: 
            return S | T
        return (S - { '' }) | T
    return S

If $\mathcal{M}$ is a set of *extended marked rules*, then the *closure* of $\mathcal{M}$ is the smallest set $\mathcal{K}$ such that
we have the following:
- $\mathcal{M} \subseteq \mathcal{K}$,
- If $a \rightarrow \beta \bullet c\, \delta: L$ is an *extended marked rule* from 
  $\mathcal{K}$, and $c$ is a variable and if, furthermore,
  $c \rightarrow \gamma$ is a grammar rule,
  then the extended marked rule $c \rightarrow \bullet \gamma: \bigcup\{\texttt{First}(\delta\,t) \mid t \in L\}$
  is an element of $\mathcal{K}$:
  $$(a \rightarrow \beta \bullet c\, \delta:L) \in \mathcal{K} 
         \;\wedge\; 
         (c \rightarrow \gamma) \in R
         \;\Rightarrow\; \Bigl(c \rightarrow \bullet \gamma: \bigcup\{\texttt{First}(\delta\,t) \mid t \in L\}\Bigr) \in \mathcal{K}
  $$

We define $\texttt{closure}(\mathcal{M}) := \mathcal{K}$.  The function `cmp_closure` computes this closure for the
given *extended marked rules* and returns it as a list.  It collects the extended marked rules in the dictionary `Result`, which is used as an
*ordered set*:  Its keys are the extended marked rules, its values are `None`, and it remembers the order in which the
extended marked rules have been inserted.  The list `worklist` contains the extended marked rules that still have to be processed.

In [ ]:
def cmp_closure(self: Grammar, Marked_Rules: Iterable[ExtendedMarkedRule]) -> list[ExtendedMarkedRule]:
    Result   = dict.fromkeys(Marked_Rules)   # a dictionary that is used as an ordered set
    worklist = list(Result)
    for emr in worklist:                     # the worklist grows while we iterate over it
        c = emr.next_var()
        if c is None:
            continue
        delta = emr.mBeta[1:]
        L     = emr.mFollow
        for rule in self.mRules:
            if rule.mVariable == c:
                newL    = frozenset({ x for t in L for x in self.first_list(delta + (t,)) })
                new_emr = ExtendedMarkedRule(c, (), rule.mBody, newL)
                if new_emr not in Result:
                    Result[new_emr] = None
                    worklist.append(new_emr)
    return list(Result)

Grammar.cmp_closure = cmp_closure
del cmp_closure

Given a set of *extended marked rules* $\mathcal{M}$ and a *grammar symbol* $X$, the function $\texttt{goto}(\mathcal{M}, X)$ 
is defined as follows:
$$\texttt{goto}(\mathcal{M}, X) := \texttt{closure}\Bigl( \bigl\{ 
   a \rightarrow \beta\, X \bullet \delta:L \bigm| (a \rightarrow \beta \bullet X\, \delta:L) \in \mathcal{M} 
   \bigr\} \Bigr).
$$
The function `goto` combines the extended marked rules of this closure that have the same core and returns the
result as a `State`.

In [ ]:
def goto(self: Grammar, state: State, x: Symbol) -> State:
    kernel = [ emr.move_dot() for emr in state if emr.symbol_after_dot() == x ]
    return State(combine_rules(self.cmp_closure(kernel)))

Grammar.goto = goto
del goto

Large grammars have many states.  Printing all of them together with the action table and the goto table would
produce several megabytes of output, which Jupyter cannot display reliably.  Therefore, the states and the tables
are only printed if there are at most `MAX_PRINTED_STATES` states.  Conflicts are always printed, and the method
`dump_parse_table`, which is defined below, writes the complete tables into a file.

In [ ]:
MAX_PRINTED_STATES = 100

The function `all_states` computes the list of all states of an *LR-parser*.  The function starts with the state
$$ \texttt{closure}\bigl(\{ \widehat{s} \rightarrow \bullet s : \$ \}\bigr) $$
and then computes new states by using the function `goto`.  The list `mStates` grows while we iterate over it:  Every
new state is appended to it and gets the next free name of the form `s`$n$.  The symbols following the dots of a state
are tried in the order in which they occur in the extended marked rules of this state.  Therefore, the states are named in
the same way in every run.  The association between the states and their names is stored in the dictionary
`mStateNames`, which is also used to check whether a state is new.

In [ ]:
def all_states(self: Grammar) -> None:
    start_rule   = ExtendedMarkedRule('ŝ', (), (self.mStart,), frozenset({'$'}))
    start_state  = State(combine_rules(self.cmp_closure([ start_rule ])))
    self.mStates = [ start_state ]
    self.mStateNames[start_state] = 's0'
    for state in self.mStates:   # the list of states grows while we iterate over it
        for x in dict.fromkeys(emr.mBeta[0] for emr in state if not emr.is_complete()):
            next_state = self.goto(state, x)
            if next_state not in self.mStateNames:
                self.mStateNames[next_state] = f's{len(self.mStates)}'
                self.mStates.append(next_state)
                print('.', end='')
    print()
    if len(self.mStates) <= MAX_PRINTED_STATES:
        print(f'All {len(self.mStates)} LR-states:')
        for state in self.mStates:
            print(f'{self.mStateNames[state]} = {state}')
    else:
        print(f'There are {len(self.mStates)} LR-states.  They are not printed, since there are more than {MAX_PRINTED_STATES}.')

Grammar.all_states = all_states
del all_states

The following function computes the *action table* and is defined as follows:
- If $\mathcal{M}$ contains an *extended marked rule* of the form $a \rightarrow \beta \bullet t\, \delta:L$
  then we have
  $$\texttt{action}(\mathcal{M},t) := \langle \texttt{shift}, \texttt{goto}(\mathcal{M},t) \rangle.$$
- If $\mathcal{M}$ contains an *extended marked rule* of the form $a \rightarrow \beta\, \bullet:L$ such that
  $a \neq \widehat{s}$ and $t \in L$, then we define
  $$\texttt{action}(\mathcal{M},t) := \langle \texttt{reduce}, a \rightarrow \beta \rangle$$
- If $\mathcal{M}$ contains the *extended marked rule* $\widehat{s} \rightarrow s \bullet:\{\$\}$, then we define 
  $$\texttt{action}(\mathcal{M},\$) := \texttt{accept}. $$
- Otherwise, we have
  $$\texttt{action}(\mathcal{M},t) := \texttt{error}. $$

The function enters the *shift* actions first, then the *accept* action, and finally the *reduce* actions.
If a *reduce* action is to be entered for a token that already has a different action, the function
reports a conflict.  As the *accept* action is entered before the *reduce* actions, this also detects a
conflict between *accept* and *reduce*.  The augmented rule $\widehat{s} \rightarrow s$ is never used for a
*reduce* action, since the parser accepts the input instead.  For example, the grammar
```
s : a | "x"
a : s
```
is ambiguous, since the derivation $s \Rightarrow a \Rightarrow s$ can be repeated arbitrarily often.  In the
state that is reached after reading an $s$, the parser can either accept or reduce with the rule $a \rightarrow s$
if the next token is `$`.

In [ ]:
def compute_action_table(self: Grammar) -> None:
    self.mActionTable = {}
    verbose = len(self.mStates) <= MAX_PRINTED_STATES
    if verbose:
        print('\nAction Table:')
    for state in self.mStates:
        stateName = self.mStateNames[state]
        actionTable: dict[Token, str | tuple[str, str] | tuple[str, GrammarRule]] = {}
        # compute shift actions
        for token in sorted(self.mTokens):
            newState = self.goto(state, token)
            if len(newState) > 0:
                newName = self.mStateNames[newState]
                actionTable[token] = ('shift', newName)
                self.mActionTable[stateName, token] = ('shift', newName)
                if verbose:
                    print(f'action("{stateName}", {token}) = ("shift", {newName})')
        # compute the accept action
        for mr in state:
            if mr == ExtendedMarkedRule('ŝ', (self.mStart,), (), frozenset({'$'})):
                actionTable['$'] = 'accept'
                self.mActionTable[stateName, '$'] = 'accept'
                if verbose:
                    print(f'action("{stateName}", $) = accept')
        # compute reduce actions
        for mr in state:
            if mr.is_complete() and mr.mVariable != 'ŝ':
                for token in sorted(mr.mFollow):
                    action1 = actionTable.get(token)
                    action2 = ('reduce', mr.to_rule())
                    if action1 is None:
                        actionTable[token] = action2
                        r = self.mRuleNames[mr.to_rule()]
                        self.mActionTable[stateName, token] = ('reduce', r)
                        if verbose:
                            print(f'action("{stateName}", {token}) = {action2}')
                    elif action1 != action2:
                        self.mConflicts = True
                        print('')
                        print(f'conflict in state {stateName}:')
                        print(f'{stateName} = {state}')
                        print(f'action("{stateName}", {token}) = {action1}')
                        print(f'action("{stateName}", {token}) = {action2}')
                        print('')

Grammar.compute_action_table = compute_action_table
del compute_action_table

The function `compute_goto_table` computes the *goto table*.

In [ ]:
def compute_goto_table(self: Grammar) -> None:
    self.mGotoTable = {}
    verbose = len(self.mStates) <= MAX_PRINTED_STATES
    if verbose:
        print('\nGoto Table:')
    for state in self.mStates:
        for var in sorted(self.mVariables):
            newState = self.goto(state, var)
            if len(newState) > 0:
                stateName = self.mStateNames[state]
                newName   = self.mStateNames[newState]
                self.mGotoTable[stateName, var] = newName
                if verbose:
                    print(f'goto({stateName}, {var}) = {newName}')

Grammar.compute_goto_table = compute_goto_table
del compute_goto_table

In [ ]:
def strip_quotes(t: Token) -> Token:
    if t[0] == '"' and t[-1] == '"':
        return t[1:-1]
    return t

In [ ]:
def dump_parse_table(self: Grammar, file: str) -> None:
    with open(file, 'w', encoding="utf-8") as handle:
        handle.write('# Grammar rules:\n')
        for rule in self.mRules:
            rule_name = self.mRuleNames[rule]
            handle.write(f'{rule_name} = ("{rule.mVariable}", {rule.mBody})\n')
        handle.write('\n# Action table:\n')
        handle.write('actionTable = {}\n')
        for s, t in self.mActionTable:
            action = self.mActionTable[s, t]
            t = strip_quotes(t)
            if action[0] == 'reduce':
                rule_name = action[1]
                handle.write(f"actionTable['{s}', '{t}'] = ('reduce', {rule_name})\n")
            elif action == 'accept':
                handle.write(f"actionTable['{s}', '{t}'] = 'accept'\n")
            else:
                handle.write(f"actionTable['{s}', '{t}'] = {action}\n")
        handle.write('\n# Goto table:\n')
        handle.write('gotoTable = {}\n')
        for s, v in self.mGotoTable:
            state = self.mGotoTable[s, v]
            handle.write(f"gotoTable['{s}', '{v}'] = '{state}'\n")
        handle.write('\n# States:\n')
        handle.write('stateTable = {}\n')
        for state, name in self.mStateNames.items():
            rules = ', '.join(repr(str(mr)) for mr in state)
            handle.write(f"stateTable['{name}'] = {{ {rules} }}\n")

Grammar.dump_parse_table = dump_parse_table
del dump_parse_table

In [ ]:
def analyse(file: str) -> None:
    rules        = parse(file)
    grammarRules = transform(rules)
    grammar      = Grammar(grammarRules)
    grammar.dump_parse_table('parse-table.py')

The file `dragon-book.g` contains the grammar from the *Dragon Book* that is used in the section on LALR parsers
in the lecture notes:
$$ s \rightarrow c\, c, \qquad c \rightarrow \texttt{"x"}\, c \mid \texttt{"y"} $$
Its canonical LR parser has 10 states.  The generator names them in the same way as the figure of the
LR-goto-graph in the lecture notes, so the output can be compared with this figure.

In [ ]:
analyse('Examples/dragon-book.g')

In [ ]:
analyse('Examples/cc.g')

In [ ]:
analyse('Examples/arith.g')

In [ ]:
analyse('Examples/arith-ambiguous.g')

In [ ]:
analyse('Examples/postfix.g')

In [ ]:
analyse('Examples/arith-small.g')

In [ ]:
%%time
analyse('Examples/c-grammar.g')